[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/stage5_project.ipynb)

# 阶段五 · 综合项目：从数据库到文字报告（Day 20）

> 《Python 数据处理 20 天学习计划》收官项目。
> **开始前：文件 → 在云端硬盘中保存一份副本。**

**项目任务**：独立完成一条完整的数据处理流程——

> 在 SQLite 建两张表 → 用 SQL 做 JOIN 和分组统计 → 用 Pandas 清洗加工 → 用 f-string 拼接生成每个班的成绩摘要报告。

**完成标准**：全流程代码能从头跑到尾不报错，输出每个班级一段文字摘要。卡住时，允许回看 Day 9、12、17–19 的内容；实在不行，再翻到文末看「参考完成示例」。

## 第 1 步 · 建库建表

运行下面的代码，建好数据库（这一步已帮你写好，读懂它即可）：

- `students` 表：学号、姓名、班级、电话（有缺失，模拟真实数据）；
- `scores` 表：学号、数学、英语、语文。

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("final_project.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS students")
cur.execute("DROP TABLE IF EXISTS scores")

cur.execute("CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT, class TEXT, phone TEXT)")
cur.execute("CREATE TABLE scores (student_id INTEGER, math REAL, english REAL, chinese REAL)")

students_data = [
    (101, "小明", "一班", "138-0000-1111"),
    (102, "小红", "二班", None),                 # 没填电话 → NULL
    (103, "小刚", "一班", "137 2222 3333"),
    (104, "小丽", "二班", "136.3333.4444"),
    (105, "小华", "一班", None),
    (106, "小芳", "二班", "135-4444-5555"),
]
scores_data = [
    (101, 88, 92, 85),
    (102, 95, 85, 90),
    (103, 62, 70, 66),
    (104, 77, 88, 82),
    (105, 85, 90, 88),
    (106, 90, 86, 91),
]
cur.executemany("INSERT INTO students VALUES (?,?,?,?)", students_data)
cur.executemany("INSERT INTO scores VALUES (?,?,?,?)", scores_data)
conn.commit()
print("数据库就绪：students 表", len(students_data), "行，scores 表", len(scores_data), "行")

## 第 2 步 · 用 SQL 取数（独立完成）

**任务**：写一条带 JOIN 的 SQL，查出每个同学的 姓名、班级、电话、数学、英语、语文，用 `pd.read_sql_query` 读成 DataFrame，存到变量 `df` 并查看。

提示骨架（把 `___` 补全）：
```python
df = pd.read_sql_query("""
SELECT s.name, s.class, s.phone, sc.math, sc.english, sc.chinese
FROM students s
JOIN scores sc ON s.___ = sc.___
""", conn)
```

In [ ]:
# 第 2 步：在这里写你的代码

## 第 3 步 · 用 Pandas 清洗加工（独立完成）

**任务**，在上一步的 `df` 上完成三件事：

1. **清洗电话**：把电话里的非数字字符去掉（提示：`.str.replace(r"\D", "", regex=True)`，缺失的电话保持缺失即可）；
2. **新增「总分」列**：数学 + 英语 + 语文；
3. **找出没填电话的同学**：提示他们补录（提示：`df["电话"].isnull()` 或 SQL 思路都可以）。

In [ ]:
# 第 3 步：在这里写你的代码

## 第 4 步 · 生成班级成绩摘要（独立完成）

**任务**：按班级分组统计，用 f-string 为每个班打印一段摘要，格式参考：

```
【一班】共 3 人：小明、小刚、小华
　全班平均总分 242.0 分，最高分为小明（265 分）。
【二班】共 3 人：小红、小丽、小芳
　全班平均总分 261.3 分，最高分为小红（270 分）。
```

提示：`df.groupby("班级")` 循环遍历分组；组内用 `df.loc[总分.idxmax()]` 找最高分同学；人名列表用 `"、".join()` 拼接。

In [ ]:
# 第 4 步：在这里写你的代码

---

## 🔍 复盘自评（完成后做）

对照下面清单，给自己打分（掌握 / 待巩固），待巩固的列入后续两周复习清单：

| 知识点 | 自评 |
|---|---|
| 字符串：切片、常用方法、f-string |  |
| Pandas：筛选、缺失值、concat / merge |  |
| Pandas：groupby 分组聚合 |  |
| 文本：.str 向量化操作、正则提取 |  |
| SQL：SELECT / WHERE / GROUP BY / JOIN |  |
| SQL ↔ Pandas：read_sql_query / to_sql |  |

**建议**：把这份复盘截图保存，和四个阶段的「问题清单」放在一起，就是一份完整的学习档案。

---

## 🚀 可选扩展（仅演示了解，不计入考核）

### 关于「简单网页 App」
像 Streamlit 这样的工具，可以把数据处理脚本变成一个网页 App——别人打开网页就能传数据、看图表。核心思路就三步：写脚本 → 加几行界面代码 → 运行。示例（了解即可，不要求运行）：

```python
# demo_app.py —— 一个迷你数据网页的完整样子（仅演示）
import streamlit as st
import pandas as pd

st.title("班级成绩查询")
df = pd.read_csv("scores.csv")
class_name = st.selectbox("选择班级", df["班级"].unique())
st.write(df[df["班级"] == class_name])
```

> 这类工具一般在自己电脑或服务器上运行；Colab 里运行需要额外配置，**知道「原来是这样做的」就够了**，时间充裕再找教程跟着做一次。

### 关于「服务器」
你在 Colab 里写的代码，其实一直运行在 Google 的云端服务器上——只是 Google 把运维都替你做了。学习阶段不需要自己购买或维护服务器，了解这个概念即可。

---

## ✅ 参考完成示例（全部独立尝试之后再看）

In [ ]:
# ===== 第 2 步参考：SQL 取数 =====
df = pd.read_sql_query("""
SELECT s.name, s.class, s.phone, sc.math, sc.english, sc.chinese
FROM students s
JOIN scores sc ON s.id = sc.student_id
""", conn)
df

In [ ]:
# ===== 第 3 步参考：清洗加工 =====
# 1. 电话只保留数字（注意先转字符串，缺失值会变成 "nan"，稍后再处理）
df["phone"] = df["phone"].str.replace(r"\D", "", regex=True)

# 2. 新增总分列
df["总分"] = df["math"] + df["english"] + df["chinese"]

# 3. 找出没填电话的同学
missing_phone = df[df["phone"].isnull()]
print("请提醒以下同学补录电话：", "、".join(missing_phone["name"]))
df

In [ ]:
# ===== 第 4 步参考：生成班级摘要 =====
for class_name, group in df.groupby("class"):
    names = "、".join(group["name"])
    avg_total = group["总分"].mean()
    top = group.loc[group["总分"].idxmax()]   # 总分最高的那一行
    print(f"【{class_name}】共 {len(group)} 人：{names}")
    print(f"　全班平均总分 {avg_total:.1f} 分，最高分为{top['name']}（{top['总分']:.0f} 分）。")

---

## 🎓 结语

到这里，20 天的内容全部完成：字符串 → Pandas → 文本进阶 → SQL → 综合项目。
这套流程（取数 → 清洗 → 加工 → 输出）就是科研和工作中数据处理的最小闭环。
后续想继续深入，可以回到网页版学习计划「参考资料」里的拓展清单：pandas 刷题库、PDSH 全书、窗口函数专题。祝学习顺利！